# Lesson 39: CrewAI project — a fraud-detection team 🕵️‍♀️💳
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Project goal:** build a system that screens card transactions for fraud by combining
**rules + machine learning + a crew of AI agents + a Flow + a human decision**.

**Learning goals**
1. Generate a realistic (synthetic) transaction dataset and **label** it
2. Build fraud signals with **rules** and an **anomaly-detection** model (Isolation Forest)
3. Wrap them as **custom tools** for a CrewAI crew: analyst → investigator → report writer
4. Orchestrate everything with a **Flow** that routes high-risk cases to a **human**
5. **Evaluate** the system and discuss fairness and false alarms

**⏱️ Timing:** data 10' | signals 15' | crew 20' | flow + human review 10' | evaluation 5'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**. Multi-agent crews make many LLM calls, so this lesson uses `qwen2.5:7b` (about 5 GB to download). For a faster but less reliable run, change it to `qwen2.5:3b`.

> 💳 All data in this notebook is **made up**. Never put real people's financial data into practice projects.

## 🏦 How banks fight fraud (simplified)
| Layer | Example | Strength | Weakness |
|---|---|---|---|
| **Rules** | "Flag purchases over $2,000 at 3am from a new country" | Clear, explainable | Fraudsters learn the rules |
| **Machine learning** | A model spots unusual patterns | Finds new tricks | Hard to explain, false alarms |
| **AI agents** | Investigate a case, gather evidence, write a report | Saves analysts time | Can be wrong — needs checking |
| **Humans** | An analyst makes the final call | Judgement, accountability | Slow, expensive |

We'll build all four.

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:7b"     # smarter at using tools. "qwen2.5:3b" downloads faster but makes more mistakes
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install crewai
import os, warnings; warnings.filterwarnings("ignore")
os.environ["CREWAI_TRACING_ENABLED"] = "false"     # don't send traces anywhere / don't ask about it
os.environ["OTEL_SDK_DISABLED"] = "true"
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model=f"ollama/{MODEL}", base_url="http://localhost:11434", temperature=0.2)
print("✅ CrewAI ready, using", MODEL)

## Part 1: Make a synthetic dataset 🧪

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
customers = {f"C{i:03d}": {"home": rng.choice(["NZ", "AU"]), "avg": rng.uniform(20, 150)} for i in range(40)}
categories = ["groceries", "transport", "restaurants", "online shopping", "pharmacy", "electronics", "gift cards", "travel"]

rows = []
for tid in range(1500):                       # normal customers… who sometimes do unusual (but innocent) things
    cid = rng.choice(list(customers)); c = customers[cid]
    travelling = rng.random() < 0.06
    big_buy = rng.random() < 0.04
    rows.append({"transaction_id": f"T{tid:05d}", "customer_id": cid,
                 "hour": int(rng.choice([0, 1, 2, 3, 23])) if rng.random() < 0.04 else int(rng.normal(14, 4)) % 24,
                 "amount": round(c["avg"] * rng.uniform(4, 10) if big_buy else abs(rng.normal(c["avg"], c["avg"] * 0.5)), 2),
                 "country": rng.choice(["AU", "US", "FJ", "GB"]) if travelling else c["home"],
                 "category": rng.choice(["electronics", "travel"]) if big_buy else rng.choice(categories[:5]),
                 "new_device": bool(rng.random() < (0.3 if travelling else 0.03)), "is_fraud": 0})
for tid in range(1500, 1560):                 # 60 frauds: most are obvious, about a third are sneaky
    cid = rng.choice(list(customers)); c = customers[cid]
    sneaky = rng.random() < 0.35
    rows.append({"transaction_id": f"T{tid:05d}", "customer_id": cid,
                 "hour": int(rng.normal(14, 3)) % 24 if sneaky else int(rng.choice([1, 2, 3, 4])),
                 "amount": round(c["avg"] * rng.uniform(1.5, 3), 2) if sneaky else round(rng.uniform(400, 3000), 2),
                 "country": c["home"] if sneaky else rng.choice(["NZ", "AU", "RU", "BR", "NG", "US"]),
                 "category": rng.choice(["online shopping", "gift cards"]) if sneaky else rng.choice(["electronics", "gift cards", "travel"]),
                 "new_device": bool(rng.random() < (0.5 if sneaky else 0.7)), "is_fraud": 1})
df = pd.DataFrame(rows).sample(frac=1, random_state=0).reset_index(drop=True)
df["home_country"] = df["customer_id"].map(lambda c: customers[c]["home"])
df["customer_avg"] = df["customer_id"].map(lambda c: round(customers[c]["avg"], 2))
print(df.shape, "| fraud rate:", f"{df.is_fraud.mean():.1%}")
df.head()

> ⚖️ **Note:** `is_fraud` is the "answer key". Our system must **not** look at it — we only use it at the end to measure how well we did.

## Part 2: Fraud signals — rules + machine learning 📐

In [ ]:
def rule_flags(t):
    flags = []
    if t["amount"] > 5 * t["customer_avg"]: flags.append("amount 5x higher than usual")
    if t["hour"] < 5: flags.append("between midnight and 5am")
    if t["country"] != t["home_country"]: flags.append(f"abroad ({t['country']})")
    if t["new_device"]: flags.append("new device")
    if t["category"] in ["gift cards", "electronics"]: flags.append(f"high-risk category ({t['category']})")
    return flags

df["rule_flags"] = df.apply(rule_flags, axis=1)
df["rule_score"] = df["rule_flags"].str.len()

from sklearn.ensemble import IsolationForest
features = pd.DataFrame({"amount_ratio": df.amount / df.customer_avg, "hour": df.hour,
                         "abroad": (df.country != df.home_country).astype(int), "new_device": df.new_device.astype(int)})
iso = IsolationForest(contamination=0.05, random_state=0).fit(features)     # learns what "normal" looks like
df["anomaly_score"] = (-iso.score_samples(features)).round(3)                # higher = stranger

df["risk"] = np.where((df.rule_score >= 3) | (df.anomaly_score > df.anomaly_score.quantile(0.97)), "HIGH",
             np.where(df.rule_score >= 2, "MEDIUM", "LOW"))
print(df.risk.value_counts())
print("\nFraud caught in HIGH:", df[(df.risk == "HIGH") & (df.is_fraud == 1)].shape[0], "of", df.is_fraud.sum())
print("Innocent customers flagged HIGH (false alarms):", df[(df.risk == "HIGH") & (df.is_fraud == 0)].shape[0])

## Part 3: Custom tools for the crew 🛠️

In [ ]:
from crewai.tools import tool
import json

@tool("List high-risk transactions")
def list_high_risk(limit: int = 5) -> str:
    """List the IDs of the highest-risk transactions to investigate, most suspicious first."""
    top = df[df.risk == "HIGH"].sort_values("anomaly_score", ascending=False).head(int(limit))
    return json.dumps(top[["transaction_id", "customer_id", "amount", "anomaly_score"]].to_dict("records"))

@tool("Get transaction details")
def get_transaction(transaction_id: str) -> str:
    """Get full details and warning flags for one transaction ID, e.g. T01523."""
    row = df[df.transaction_id == transaction_id.strip()]
    if row.empty: return "No such transaction"
    t = row.iloc[0]
    return json.dumps({"id": t.transaction_id, "customer": t.customer_id, "amount": t.amount, "usual_amount": t.customer_avg,
                       "hour": int(t.hour), "country": t.country, "home_country": t.home_country, "category": t.category,
                       "new_device": bool(t.new_device), "rule_flags": t.rule_flags, "anomaly_score": float(t.anomaly_score)})

@tool("Get customer history")
def customer_history(customer_id: str) -> str:
    """Summarise a customer's normal spending: number of transactions, typical amount, usual hours, countries and categories."""
    h = df[(df.customer_id == customer_id.strip()) & (df.risk == "LOW")]
    if h.empty: return "No normal history found"
    return json.dumps({"customer": customer_id, "n_normal_transactions": len(h), "median_amount": float(h.amount.median()),
                       "usual_hours": f"{int(h.hour.quantile(0.1))}–{int(h.hour.quantile(0.9))}",
                       "countries": h.country.unique().tolist(), "top_categories": h.category.value_counts().head(3).index.tolist()})

print(get_transaction.run(transaction_id=df[df.risk == "HIGH"].iloc[0].transaction_id))

## Part 4: The fraud crew 👥
| Agent | Job | Tools |
|---|---|---|
| **Transaction Analyst** | Picks the most suspicious transactions and gathers facts | list, details |
| **Fraud Investigator** | Compares each case to the customer's normal behaviour and rates it | details, history |
| **Report Writer** | Writes a clear report for a human analyst, in a fixed structure | — |

In [ ]:
from pydantic import BaseModel, Field

class Case(BaseModel):
    transaction_id: str
    risk_level: str = Field(description="HIGH, MEDIUM or LOW")
    evidence: list[str] = Field(description="the specific facts that support the risk level")
    recommended_action: str = Field(description="e.g. block card and call customer, send SMS check, no action")

class FraudReport(BaseModel):
    summary: str
    cases: list[Case]

analyst = Agent(role="Transaction Analyst", goal="Find the most suspicious transactions and collect their facts",
                backstory="You work in a bank's fraud team. You always use tools for facts and never invent numbers.",
                tools=[list_high_risk, get_transaction], llm=llm, verbose=True)
investigator = Agent(role="Fraud Investigator",
                     goal="Decide how likely each case is to be fraud by comparing it with the customer's normal behaviour",
                     backstory="You are experienced and fair. Unusual is not the same as fraudulent — travellers and big "
                               "one-off purchases are common. You explain your reasoning with evidence.",
                     tools=[get_transaction, customer_history], llm=llm, verbose=True)
writer = Agent(role="Report Writer", goal="Write clear, structured fraud reports for human analysts",
               backstory="You turn investigation notes into short, factual reports.", llm=llm, verbose=True)

find = Task(description="Use your tools to list the top {n} high-risk transactions and get the details of each one.",
            expected_output="For each transaction: its ID and key facts", agent=analyst)
investigate = Task(description="For each transaction found, check the customer's history and decide a risk level "
                               "(HIGH/MEDIUM/LOW) with evidence and a recommended action.",
                   expected_output="A risk level, evidence and action for each transaction", agent=investigator, context=[find])
report = Task(description="Write the final fraud report from the investigation.", expected_output="A FraudReport",
              agent=writer, context=[investigate], output_pydantic=FraudReport)

fraud_crew = Crew(agents=[analyst, investigator, writer], tasks=[find, investigate, report], process=Process.sequential)
result = fraud_crew.kickoff(inputs={"n": 3})

In [ ]:
fr = result.pydantic
if fr:
    print("📋", fr.summary, "\n")
    for c in fr.cases:
        truth = df.loc[df.transaction_id == c.transaction_id, "is_fraud"]
        label = ("really fraud ✅" if truth.iloc[0] else "actually innocent ⚠️") if len(truth) else "unknown ID ❓"
        print(f"{c.transaction_id}: {c.risk_level:6} → {c.recommended_action}   [{label}]")
        for e in c.evidence: print("     •", e)
else:
    print(result.raw)

### 🧩 Exercise 1
1. Check each case against the evidence. Did any agent **invent** a fact that isn't in the tool output?
2. The answer key shows whether each case was really fraud. Did the crew get it right?

## Part 5: A Flow with a human in the loop 🙋
Agents shouldn't block someone's bank card on their own. Our Flow: **score** → if there are high-risk cases, **run the crew** → a **human** approves or rejects each recommended action.

In [ ]:
from crewai.flow.flow import Flow, start, listen, router

class ScreeningState(BaseModel):
    n_high: int = 0
    report: FraudReport | None = None
    decisions: dict = {}

class FraudScreeningFlow(Flow[ScreeningState]):
    @start()
    def score_transactions(self):
        self.state.n_high = int((df.risk == "HIGH").sum())
        print(f"📊 {self.state.n_high} high-risk transactions today")

    @router(score_transactions)
    def any_high_risk(self):
        return "investigate" if self.state.n_high > 0 else "all_clear"

    @listen("investigate")
    def run_crew(self):
        self.state.report = fraud_crew.kickoff(inputs={"n": 2}).pydantic

    @listen(run_crew)
    def human_review(self):
        if not self.state.report: return
        for c in self.state.report.cases:
            answer = input(f"\n{c.transaction_id} ({c.risk_level}): {c.recommended_action}\nEvidence: {c.evidence}\nApprove? (y/n) ")
            self.state.decisions[c.transaction_id] = "APPROVED" if answer.lower().startswith("y") else "REJECTED by human"

    @listen("all_clear")
    def nothing_to_do(self):
        print("✅ No high-risk transactions today.")

screening = FraudScreeningFlow()
screening.kickoff()
print("\n🧾 Final decisions:", screening.state.decisions)

## Part 6: Evaluate the whole system 📏

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
pred = (df.risk == "HIGH").astype(int)
tn, fp, fn, tp = confusion_matrix(df.is_fraud, pred).ravel()
print(f"Caught fraud (true positives):        {tp}")
print(f"Missed fraud (false negatives):       {fn}")
print(f"False alarms on innocent customers:   {fp}")
print(f"Precision (flags that were fraud):    {precision_score(df.is_fraud, pred):.0%}")
print(f"Recall (fraud that we caught):        {recall_score(df.is_fraud, pred):.0%}")

### 🧩 Exercise 2: Tune and reflect
1. Change the HIGH threshold (e.g. `rule_score >= 2`). What happens to **recall** and **false alarms**? A bank must choose: annoy innocent customers, or miss more fraud?
2. Our rules flag anyone **abroad**. Who might be unfairly flagged? (Travellers, people with family overseas, migrants…) How could you make the system fairer?
3. Where would you put a human in the loop, and why? What should agents **never** do automatically?

### 🚀 Challenge
- Add a fourth agent, a **Customer Communications** agent, that drafts a polite SMS asking the customer to confirm a transaction — only for cases the human approved.
- Turn the project into a CLI project with `crewai create flow fraud_screening` (Lesson 38).

## 📌 Summary
You built a **layered** fraud system: **rules** (explainable) + **anomaly detection** (finds the unusual) + a **crew** (investigates and reports with structured output) + a **Flow** (controls the process) + a **human** (makes the final decision) — and measured it with **precision and recall**.